In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, RandomizedSearchCV
import matplotlib.pyplot as plt
import xgboost as xgb
from datetime import timedelta
import plotly.graph_objs as go
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import uniform, randint
from sklearn.metrics import mean_squared_error
import cloudpickle

In [ ]:
def calculate_current_month(row):
    cohort_start = row['cohort']
    elapsed_months = row['elapsed_months']
    current_month = (cohort_start + pd.DateOffset(months=int(elapsed_months))).month
    return current_month

In [ ]:
def quantile_loss(alpha):
    def custom_loss(y_true, y_pred):
        errors = y_pred - y_true
        grad = np.where(errors < 0, -alpha, 1 - alpha)  # Gradient
        hess = np.ones_like(y_true)  # Hessian
        return grad, hess
    return custom_loss

In [ ]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer, mean_pinball_loss
def model_train_save_pipeline(game,alpha,X_train,y_train,X_test,y_test):
    model = xgb.XGBRegressor(objective=quantile_loss(alpha), enable_categorical=True)
    numerical_columns = X_train.select_dtypes(include=['int64', 'float64']).columns
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numerical_columns),
        ],
        remainder='passthrough'
    )
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    param_dist = {
        'model__n_estimators': randint(50, 500),
        'model__learning_rate': uniform(0.01, 0.09),
        'model__max_depth': randint(3, 10),
        'model__subsample': uniform(0.6, 0.4),
        'model__colsample_bytree': uniform(0.6, 0.4),
        'model__min_child_weight': randint(1, 10),
        'model__gamma': uniform(0, 0.3),
        'model__reg_alpha': uniform(0, 1),
        'model__reg_lambda': uniform(1, 2),
    }
    random_search = RandomizedSearchCV(
        pipeline, param_distributions=param_dist,
        n_iter=100, scoring=make_scorer(mean_pinball_loss, alpha=alpha, greater_is_better=False),
        cv=TimeSeriesSplit(n_splits=5), verbose=1, n_jobs=-1, random_state=42
    )
    random_search.fit(X_train, y_train)
    best_model1 = random_search.best_estimator_
    with open(f'{game}_xgb_{alpha}.pkl', 'wb') as f:
        cloudpickle.dump(best_model1, f)
    y_train_pred = best_model1.predict(X_train)
    y_test_pred = best_model1.predict(X_test)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
    print(f"{game} {alpha}Train RMSE: {rmse_train}")
    print(f"{game} {alpha}Test RMSE: {rmse_test}")
    return best_model1


In [ ]:
def preprocess(game):
    df = pd.read_csv(f'Latest Data/Segregated/{game}_Dataset_12_2024.csv')
    df['elapsed_months'] -= 1
    df = df[~((df['client_actual_spend'].fillna(0) == 0) & (df['total_cac'].fillna(0) == 0))]
    df['total_cac'] = df.apply(
        lambda row: row['client_actual_spend'] 
        if (pd.isnull(row['total_cac']) or row['total_cac'] == 0) and not pd.isnull(row['client_actual_spend']) 
        else row['total_cac'], 
        axis=1
    )

    df['client_actual_spend'] = df.apply(
        lambda row: row['total_cac'] 
        if (pd.isnull(row['client_actual_spend']) or row['client_actual_spend'] == 0) and not pd.isnull(row['total_cac']) 
        else row['client_actual_spend'], 
        axis=1
    )
    cl1 = df.loc[:, df.columns.str.startswith('MC_')].columns.tolist()
    df = df[~df['cohort'].str.endswith(( '2021', '2022','2020','2019','2018'))]
    df = df[~df['cohort'].str.startswith(( '2021','2022', '2020','2019','2018'))]
    columns_to_keep = ['game_id', 'total_cac', 'client_actual_spend','cohort', 'elapsed_months', 'transaction_period_ltv_cac','transaction_period']+cl1
    df = df[columns_to_keep]
    df = df.reset_index(drop=True)
    df = df[df['game_id'] == game]
    df['cohort'] = pd.to_datetime(df['cohort'], format='%d/%m/%Y')
    df['cohort_month'] = df['cohort'].dt.month
    df.fillna(0,inplace=True)
    df['current_month'] = df.apply(calculate_current_month, axis=1)
    df['cohort_month'] = df['cohort_month'].astype('category')
    df['current_month'] = df['current_month'].astype('category')
    feature_columns = [
        'total_cac', 'cohort_month', 'current_month']+cl1
    for lag in range(1, 4):
        df[f'ltv_lag_{lag}'] = df.groupby('cohort')['transaction_period_ltv_cac'].shift(lag)
    return df,feature_columns

In [ ]:
def forecast_cumulative_ltv_cac_variations(game_id, best_model_expected, best_model_bad, best_model_good, df, forecast_months=12):
    game_data = df[df['game_id'] == game_id].copy()
    forecasted_results = []
    sp_ltv_cac= df[df['transaction_period_ltv_cac'] > 0]['transaction_period_ltv_cac'].min()
    for cohort in game_data['cohort'].unique():
        cohort_data = game_data[game_data['cohort'] == cohort].copy()
        total_cac = cohort_data['total_cac'].iloc[0]
        last_elapsed_month = int(cohort_data['elapsed_months'].max())
        cumulative_actual = 0

        # Calculate cumulative actual values until the last known elapsed month
        actual_cumulative = []
        for month in range(0, last_elapsed_month + 1):
            transactional_actual = cohort_data.loc[cohort_data['elapsed_months'] == month, 'transaction_period_ltv_cac'].values
            transactional_actual = transactional_actual[0] if len(transactional_actual) > 0 else 0
            cumulative_actual += transactional_actual
            actual_cumulative.append({
                'month': month,
                'transactional_actual': transactional_actual,
                'cumulative_actual': cumulative_actual,
                'absolute_revenue_actual': total_cac * transactional_actual,
                'total_cac': total_cac
            })

        # # Initialize lag features using the last three transactional actuals
        # def get_lag_value(month_offset):
        #     elapsed_month = month_offset
        #     if elapsed_month >= 3:
        #         lag_values = cohort_data.loc[cohort_data['elapsed_months'] == elapsed_month, 'transaction_period_ltv_cac'].values
        #         return lag_values[0] if len(lag_values) > 0 else 0
        #     else:
        #         return 0

        lag_1 = np.nan
        lag_2 = np.nan
        lag_3 = np.nan
        lag_1b = np.nan
        lag_1g = np.nan

        cumulative_values = {
            'cumulative_bad': 0,
            'cumulative_expected': 0,
            'cumulative_good': 0
        }
        # Use static features from any month (e.g., last available month)
        static_features = cohort_data.loc[cohort_data['elapsed_months'] == last_elapsed_month, best_model_expected.feature_names_in_].iloc[0].copy()

        # Remove features that need to change: 'elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3'
        for col in ['elapsed_months', 'current_month', 'ltv_lag_1', 'ltv_lag_2', 'ltv_lag_3']:
            if col in static_features:
                static_features = static_features.drop(col)

        # Predict for each month up to the last actual month + forecast period
        for i in range(0, last_elapsed_month + forecast_months + 1):
            elapsed_months = i

            # Prepare features
            features = static_features.copy()

            # Update 'elapsed_months'
            features['elapsed_months'] = elapsed_months

            # Update 'current_month' if it is a feature
            if 'current_month' in best_model_expected.feature_names_in_:
                # Calculate 'current_month' based on cohort start date
                cohort_start_date = pd.to_datetime(cohort_data['cohort'].iloc[0])
                current_date = cohort_start_date + pd.DateOffset(months=elapsed_months)
                features['current_month'] = current_date.month  # Month as integer from 1 to 12
            else:
                current_date = pd.to_datetime(cohort_data['cohort'].iloc[0]) + pd.DateOffset(months=elapsed_months)

            # Update lag features
            features1 = features.copy()
            features2 = features.copy()
            

            features['ltv_lag_1'] = lag_1
            features1['ltv_lag_1'] = lag_1b
            features2['ltv_lag_1'] = lag_1g
            features['ltv_lag_2'] = lag_2
            features1['ltv_lag_2'] = lag_2
            features2['ltv_lag_2'] = lag_2
            features['ltv_lag_3'] = lag_3
            features1['ltv_lag_3'] = lag_3
            features2['ltv_lag_3'] = lag_3

            # Ensure correct format for model prediction
            features = pd.DataFrame([features])
            features1 = pd.DataFrame([features1])
            features2 = pd.DataFrame([features2])

            # Predictions for each model
            predicted_val_bad = max(0, best_model_bad.predict(features1)[0])  
            predicted_val_expected = max(0, best_model_expected.predict(features)[0])
            predicted_val_good = max(0, best_model_good.predict(features2)[0])
            x = [predicted_val_bad,predicted_val_expected,predicted_val_good]
            x.sort()
            predicted_val_bad = x[0]
            predicted_val_expected = x[1] 
            predicted_val_good = x[2]

        

            if(predicted_val_good<=0):
                predicted_val_good=sp_ltv_cac
            if(predicted_val_expected<=0):
                predicted_val_expected = predicted_val_good/2
            if(predicted_val_bad<=0):
                predicted_val_bad = predicted_val_expected/2





            # Update lag features for next month
            lag_3 = lag_2
            lag_2 = lag_1
            lag_1 = predicted_val_expected
            lag_1b = predicted_val_bad
            lag_1g = predicted_val_good
            #lag_1 = predicted_val_expected if i > last_elapsed_month else cohort_data.loc[cohort_data['elapsed_months'] == i, 'transaction_period_ltv_cac'].values[0]

            # Prepare result row
            row_result = {
                'game_id': game_id,
                'cohort': cohort,
                'elapsed_months': elapsed_months,
                'current_month': current_date.strftime('%Y-%m'),
            }

            # Add actuals if within actual data range
            actual_data = next((ac for ac in actual_cumulative if ac['month'] == elapsed_months), None)
            row_result['transactional_actual'] = actual_data['transactional_actual'] if actual_data else None
            row_result['cumulative_actual'] = actual_data['cumulative_actual'] if actual_data else None
            row_result['absolute_revenue_actual'] = actual_data['absolute_revenue_actual'] if actual_data else None
            row_result['total_cac'] = actual_data['total_cac'] if actual_data else total_cac

            # Store predictions and cumulative values
            for model_type, predicted_val in zip(
                ['bad', 'expected', 'good'],
                [predicted_val_bad, predicted_val_expected, predicted_val_good]
            ):
                cumulative_key = f"cumulative_{model_type}"
                cumulative_values[cumulative_key] += predicted_val
                row_result[f"transactional_{model_type}"] = predicted_val
                row_result[cumulative_key] = cumulative_values[cumulative_key]
                row_result[f"absolute_revenue_{model_type}"] = total_cac * predicted_val

            forecasted_results.append(row_result)

    forecast_df = pd.DataFrame(forecasted_results)
    return forecast_df


In [ ]:
games = ['G001','G030','G034']

In [ ]:
for game in games:
    df,feature_columns = preprocess(game)
    # X = df[feature_columns + ['elapsed_months']]
    # y = df['transaction_period_ltv_cac']
    X = df[feature_columns + ['elapsed_months']].drop(df.index[-1])
    y = df['transaction_period_ltv_cac'].drop(df.index[-1])
    # Time-based split (fix C2): train on earlier transaction months, test on the most recent 20%
    _tp = pd.to_datetime(df.loc[X.index, 'transaction_period'], format='%d/%m/%Y', errors='coerce')
    _order = _tp.sort_values(kind='stable').index
    _cut = int(len(_order) * 0.8)
    X_train, X_test = X.loc[_order[:_cut]], X.loc[_order[_cut:]]
    y_train, y_test = y.loc[_order[:_cut]], y.loc[_order[_cut:]]
    alphas=[0.25,0.5,0.75]
    model = []
    for alpha in alphas:
        m = model_train_save_pipeline(game,alpha,X_train,y_train,X_test,y_test)
        model.append(m)
    df = df[~df['cohort'].astype(str).str.contains('|'.join(['2021', '2020', '2022', '2019', '2018']))]
    df2 = forecast_cumulative_ltv_cac_variations(game, model[1],model[2],model[0], df, 12)
    df2.to_csv(f'{game}_predictions.csv',index=False)    